# 01 · Sensor data and your first neural network

**Goals of this notebook**

1. Understand what the motion sensor (IMU) measures.
2. Look at gesture data and prepare it for machine learning.
3. Learn how a neural network works and how it is trained.
4. Train the **baseline model**: the uncompressed model that notebooks 02-05 will shrink.
5. Compute what this model would cost on the Arduino.
6. *(Optional, with a board)* Record your own gestures.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

from aiedge import config as C
from aiedge.data import load_gesture_data
from aiedge.models import MLP, count_params, count_macs
from aiedge.training import train, accuracy, set_seed, predict_logits, confusion_matrix
from aiedge.export import mcu_cost
from aiedge import plots

## 1. What does the IMU measure?

An **IMU** (Inertial Measurement Unit) contains two sensors we use:

**The accelerometer** measures acceleration along three axes (x, y, z), in units of **g** (1 g = 9.81 m/s², the acceleration of gravity).
* A board lying still on a table reads about **(0, 0, +1 g)**. That may be surprising: it is not moving! But the accelerometer feels gravity: the table pushes the board *up* with a force that would accelerate it at 1 g. So a resting accelerometer tells you **which way is up**.
* When you move the board, the motion's acceleration is added on top: a punch forward shows a big positive spike in x (speeding up), then a negative one (stopping).

**The gyroscope** measures **how fast the board rotates** around each axis, in degrees per second (°/s). A board lying still reads (0, 0, 0). Twisting your wrist gives a big signal around the x axis.

Our board-axis convention (hold the board flat, components up, USB cable towards your wrist):

```
          +z (up, out of the components side)
           |
           |____ +x (forward, away from you: the punch direction)
          /
        +y (to your left)
```

### From continuous motion to numbers

The board reads the IMU **50 times per second** (the *sampling rate*, 50 Hz). Each reading has 6 numbers: `ax, ay, az` (g) and `gx, gy, gz` (°/s). A gesture takes about a second, so we cut out a **window** of 50 readings, which gives a table of 50 rows × 6 columns = **300 numbers** per gesture.

When does a window start? The board constantly checks whether it is moving: whether the acceleration differs from 1 g by more than 0.35 g, or whether it rotates faster than 120 °/s. That moment is the *trigger*. The board also keeps the last 10 readings (0.2 s) in memory, so the window includes the start of the movement.

## 2. The data

We generate data with the physics simulator in `aiedge/data.py`. For each example it picks a gesture and then randomises *how* it is performed: speed, size, direction, how the board is tilted in the hand, a bit of arm drift and sensor noise. That's why no two examples look the same. This variation is exactly what makes the problem hard (and realistic).

`load_gesture_data()` generates 800 examples per gesture, splits them, and prepares them for training (more on that below).

In [ ]:
data = load_gesture_data(source="synthetic")
print(data.summary())

Let's look at a few examples: one column per gesture, two rows per example (accelerometer on top, gyroscope below).

In [ ]:
plots.plot_examples(data.raw_train, data.y_train.numpy(), per_class=2, seed=1)
plt.show()

**Look carefully and answer for yourself:**
* In which plot can you see gravity? (Hint: which axis sits around +1 g when nothing happens?)
* Why is `punch` mostly visible in `ax`, and `lift` in `az`?
* Which gesture shows mostly in the gyroscope?
* Two examples of the same gesture can look quite different. Why?

You can probably tell the gestures apart by eye most of the time. The network must learn to do the same thing, from examples alone.

## 3. Preparing data for machine learning

### Features and labels
In **supervised learning** every example is a pair:
* the input, called **features** `x`: here the 300 sensor numbers;
* the answer, called the **label** `y`: here the gesture, encoded as a number: 0 = punch, 1 = lift, 2 = wave, 3 = circle, 4 = twist.

The model must learn a function `x → y` from examples.

### Training, validation and test sets
We split the examples into three groups (70% / 15% / 15%):

| Set | Used for | Analogy |
|---|---|---|
| **training set** | the model learns from these | the exercises in your textbook |
| **validation set** | we check progress during training and choose settings | a practice exam |
| **test set** | measured **once**, at the very end | the real exam |

Why not test on the training data? A model can *memorise* its training examples. That is called **overfitting**: it looks perfect on data it has seen and fails on new data. Only data the model has never seen tells us how well it will work on your board.

### Normalisation
The accelerometer values are around ±1-4 g; the gyroscope values around ±100s of °/s. Neural networks learn much better when all inputs have a similar scale. So for each of the 6 channels we subtract its mean and divide by its standard deviation:

$$x_{\text{norm}} = \frac{x - \text{mean}}{\text{std}}$$

Important: mean and std are computed on the **training set only** and then applied to everything, including the data the board will measure later. The board needs these 12 numbers too, so notebook 05 copies them into the C++ code.

In [ ]:
print("mean per channel:", dict(zip(C.CHANNELS, data.channel_mean.round(3))))
print("std  per channel:", dict(zip(C.CHANNELS, data.channel_std.round(3))))
print()
print("X_train:", tuple(data.X_train.shape), " <- (number of examples, 300 features)")
print("y_train:", tuple(data.y_train.shape), " first labels:", data.y_train[:10].tolist())

## 4. How a neural network works

### One neuron
A neuron takes some inputs $x_1, \dots, x_n$, multiplies each by a **weight** $w_i$, adds them up together with a **bias** $b$, and passes the result through an **activation function**:

$$\text{output} = \text{ReLU}\Big(\sum_{i=1}^{n} w_i x_i + b\Big) \qquad \text{ReLU}(z) = \max(0, z)$$

The weights say how important each input is (and in which direction); the bias shifts the result. **ReLU** ("rectified linear unit") simply replaces negative numbers by 0. This non-linearity is what lets networks learn more than straight lines.

Each "multiply a weight by an input and add it to a running sum" is called a **multiply-accumulate (MAC)**. Remember that word: MACs are the main cost of running a network on a microcontroller.

Let's compute one neuron by hand:

In [ ]:
x = np.array([0.5, -1.0, 2.0])        # 3 inputs
w = np.array([0.8, 0.2, -0.5])        # 3 weights
b = 0.1                                # bias

z = np.sum(w * x) + b                  # 3 MACs + bias
print(f"weighted sum: 0.8*0.5 + 0.2*(-1.0) + (-0.5)*2.0 + 0.1 = {z:.2f}")
print(f"after ReLU:   {max(0.0, z):.2f}")

### Layers and the multi-layer perceptron (MLP)
A **layer** is a group of neurons that all receive the same inputs. Stacking layers gives a **multi-layer perceptron** (MLP), also called a *fully connected* network:

```
 input (300)  ->  hidden layer 1 (64 neurons)  ->  hidden layer 2 (32)  ->  output layer (5)
                   ReLU                              ReLU                    no activation
```

Every neuron is connected to every neuron of the previous layer. Think of the network as a **graph**: the neurons are the **nodes** and the weights are the **edges** between them. (Notebook 03 will remove edges and nodes: *edge pruning* and *node pruning*.)

For a layer with $n_{in}$ inputs and $n_{out}$ neurons, all the weights form a matrix $W$ of size $n_{out} \times n_{in}$, and the layer computes $y = \text{ReLU}(W x + b)$.

* number of parameters = $n_{in} \cdot n_{out}$ weights $+\ n_{out}$ biases
* number of MACs = $n_{in} \cdot n_{out}$

### The output: logits and softmax
The last layer has one neuron per gesture. Its outputs are called **logits**: raw scores, the higher the more likely. The **softmax** function turns them into probabilities that add up to 1:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

The prediction is the gesture with the highest probability.

In [ ]:
set_seed(0)                      # same random start every time -> reproducible results
model = MLP(hidden=(64, 32))     # 300 -> 64 -> 32 -> 5
print(model)
for i, layer in enumerate(model.layers):
    n_in, n_out = layer.in_features, layer.out_features
    print(f"layer {i}: {n_in:3d} -> {n_out:2d}   weights {n_in * n_out:6,}  + biases {n_out:2d}")
print(f"total parameters: {count_params(model):,}   MACs per prediction: {count_macs(model):,}")

The weights start as small *random* numbers, so the untrained network is just guessing. With 5 gestures, guessing gives about 20% accuracy:

In [ ]:
logits = predict_logits(model, data.X_test[:3])
print("logits of 3 test examples:\n", logits.numpy().round(2))
print("probabilities:\n", torch.softmax(logits, dim=1).numpy().round(2))
print(f"accuracy before training: {accuracy(model, data.X_test, data.y_test):.1%}")

## 5. How a neural network learns

Training means: **find the weights that make the predictions match the labels.**

1. **Loss function**: a number that says how wrong the network is. For classification we use **cross-entropy**: $\text{loss} = -\log(p_{\text{correct class}})$. If the network gives the right gesture probability 0.9, the loss is 0.105; if it gives it only 0.1, the loss is 2.3. Lower is better.
2. **Gradient**: for every weight, how much the loss would change if we nudged that weight a little. PyTorch computes all gradients automatically with **backpropagation** (the chain rule from calculus, applied layer by layer from the output back to the input).
3. **Gradient descent**: move every weight a small step *against* its gradient, so the loss goes down:
   $$w \leftarrow w - \text{learning rate} \times \frac{\partial\, \text{loss}}{\partial w}$$
   The **learning rate** sets the step size: too large and training jumps around, too small and it takes forever. We use **Adam**, a popular variant that adapts the step size for each weight.
4. Repeat. We do not compute the loss over all 2800 training examples at once, but over small random **mini-batches** of 64 examples. One pass over the whole training set is an **epoch**.

### Early stopping
After every epoch we measure accuracy on the **validation set**. Training accuracy usually keeps going up, but at some point validation accuracy stops improving: the network starts to memorise instead of learning general patterns (overfitting). We keep the weights from the best validation epoch and stop when there has been no improvement for 25 epochs.

### Data augmentation
On the board, the trigger never fires at exactly the same moment within a movement. So during training we randomly shift every example a few readings (up to ±0.12 s) earlier or later. This is **data augmentation**: creating extra, realistic variations of the training data for free. It teaches the network that a gesture is the same gesture whether it starts a little earlier or later.

Let's train! (This takes a few seconds to a minute.)

In [ ]:
set_seed(0)
baseline = MLP(hidden=(64, 32))
history = train(baseline, data, augment=True)

In [ ]:
plots.plot_history(history, "Baseline 300-64-32-5")
plt.show()

**Reading the curves**: the loss drops quickly in the first epochs and then flattens. Training accuracy ends higher than validation accuracy. That gap is a small amount of overfitting, which is normal.

## 6. Evaluating the baseline
Now, *once*, the real exam: the test set.

In [ ]:
test_acc = accuracy(baseline, data.X_test, data.y_test)
print(f"baseline test accuracy: {test_acc:.1%}")

y_pred = predict_logits(baseline, data.X_test).argmax(1).numpy()
cm = confusion_matrix(data.y_test.numpy(), y_pred)
plots.plot_confusion(cm, f"Baseline: test accuracy {test_acc:.1%}")
plt.show()

**The confusion matrix** shows, for each true gesture (row), what the network predicted (column). The diagonal holds the correct answers; everything off the diagonal is a mistake. Which gestures does the network confuse most? Does that make physical sense?

Let's look at some mistakes:

In [ ]:
wrong = np.flatnonzero(y_pred != data.y_test.numpy())
print(f"{len(wrong)} mistakes out of {len(y_pred)} test examples")
fig, axes = plt.subplots(2, 3, figsize=(12, 5), sharex=True)
for col, i in enumerate(wrong[:3]):
    title = f"true: {C.GESTURES[data.y_test[i]]}, predicted: {C.GESTURES[y_pred[i]]}"
    plots.plot_gesture(data.raw_test[i], title, axes=axes[:, col])
fig.tight_layout()
plt.show()

### How precise is "test accuracy"?
Our test set has 600 examples, so one mistake more or less changes the accuracy by 0.17 percentage points. Statistics tells us that with ~95% accuracy on 600 examples, the measurement itself is only reliable to about **±1 percentage point**. Training with a different random seed also moves the result by around half a point.

So in the next notebooks: **do not get excited about differences smaller than ~1 point.** Look for clear trends.

## 7. What would this model cost on the Arduino?

Every weight and bias is a 32-bit floating-point number (**float32**), which takes 4 bytes. The weights are stored in **flash**. The activations (the intermediate results) live in **RAM** while the network runs. The time is dominated by the MACs: a plain C loop on the Cortex-M4F needs very roughly 4 clock cycles per MAC (load the weight, load the input, multiply-add, loop), and the clock runs at 64 million cycles per second.

In [ ]:
cost = mcu_cost(baseline)
print(f"parameters:            {cost['params']:,}")
print(f"flash for the weights: {cost['flash_KB']:.1f} KB  ({cost['flash_%']:.1f}% of the 1 MB flash)")
print(f"RAM for activations:   {cost['RAM_KB']:.2f} KB")
print(f"MACs per prediction:   {cost['MACs']:,}")
print(f"estimated time:        {cost['est_ms']:.2f} ms per prediction (rough estimate, measured for real in notebook 05)")

Good news: this model would already fit on the board. **So why compress it?**

* **The rest of the program needs room too.** The Arduino core (USB, timers, the operating system underneath) already takes ~95 KB of flash and ~47 KB of RAM before your code does anything. Bluetooth, logging and other sensors each want their share.
* **Energy.** Every MAC and every byte read from memory costs energy. A model that is 10x cheaper can make a battery last much longer.
* **Cheaper hardware.** Many microcontrollers have 32-128 KB of flash and no FPU. A small integer model runs there too.
* **Bigger problems.** Recognising spoken words or images needs networks of hundreds of kilobytes to megabytes. The same techniques decide whether those fit or not.

### 🎯 The course challenge
> Build a model for the board that uses **at most 10 KB of memory for its parameters** (8x less than the baseline!) while losing **at most 1 percentage point of accuracy** compared to the baseline.

Write down your baseline accuracy. You will need it in notebook 05.

In [ ]:
models_dir = ROOT / "models"
models_dir.mkdir(exist_ok=True)
torch.save(baseline.state_dict(), models_dir / "baseline.pt")
print(f"baseline test accuracy = {test_acc:.1%}, saved to {models_dir / 'baseline.pt'}")

## 8. (Optional) Record your own gestures

A model trained on *your* recordings, made with *your* board, will work best on your board. You need the board and about 15 minutes.

**Step 1: upload the capture sketch.**
Open `arduino/gesture_capture/gesture_capture.ino` in the Arduino IDE. Set `#define BOARD_REVISION` to 1 or 2 at the top, then upload. Open the Serial Monitor (115200 baud) and move the board: every movement prints 50 lines of numbers. **Close the Serial Monitor afterwards** (only one program can use the port at a time).

**Step 2: record.** Set `RECORD = True` in the next cell and run it once for each gesture. Record about **40 repetitions of each gesture**. Tips:
* Hold the board the way it is described in notebook 00 and the way you will hold it later.
* Pause about a second between repetitions; the orange LED is on while the board records.
* Vary a little: speed, size. Ask friends to record too: more variety means a model that works for everyone.
* If one recording went wrong, don't worry; a few bad examples do little harm.

In [ ]:
RECORD = False            # set to True when the board runs gesture_capture
GESTURE = "punch"         # one of: punch, lift, wave, circle, twist
COUNT = 40

if RECORD:
    from aiedge.board import record_gestures, list_ports
    list_ports()
    record_gestures(GESTURE, COUNT)          # add port="COM5" or "/dev/ttyACM0" if it is not found
else:
    print("Recording is switched off (RECORD = False).")

**Step 3: train on your data.** When `data/recorded/` contains a file for each gesture, load the recordings instead of the synthetic data and re-run the training cells above (sections 3-6):

```python
data = load_gesture_data(source="recorded")
```

With only ~40 examples per gesture your test set is small (~30 examples), so the accuracy will jump around more (see "How precise is test accuracy?"). Notebook 05 has the same switch: set `DATA_SOURCE = "recorded"` there to deploy a model trained on your own gestures.

In [ ]:
recorded = sorted((ROOT / "data" / "recorded").glob("*.csv"))
print("recordings found:", [p.name for p in recorded] or "none yet")

## Summary

* The IMU gives 6 numbers 50 times per second; one gesture = 300 numbers.
* A neural network is a stack of layers; each neuron computes a weighted sum, adds a bias and applies ReLU. Each weight costs one **MAC** per prediction.
* Training = gradient descent on the cross-entropy loss, in mini-batches. Early stopping on the validation set prevents overfitting; data augmentation adds realistic variation.
* Our **baseline** (300-64-32-5) has ~21 500 parameters (~84 KB in float32) and needs ~21 400 MACs per prediction.
* **Challenge:** ≤ 10 KB, at most 1 point less accurate.

## Exercises

1. ⭐ **Model size.** Train MLPs with `hidden=(8,)`, `(16, 8)`, `(64, 32)` and `(256, 128)`. Make a table: parameters, KB (float32), test accuracy. Is bigger always better? Why might the biggest network not win?
2. ⭐ **Count by hand.** Compute the number of parameters and MACs of a 300-16-8-5 network on paper, then check with `count_params` and `count_macs`.
3. **Learning rate.** Train the baseline with `lr=0.03` and `lr=0.0003` (argument of `train`). Plot the histories. What happens?
4. **No augmentation.** Train with `augment=False`. Does test accuracy change? Why could the effect on the *real board* be bigger than on this test set?
5. **Seeds.** Train the baseline 3 times with `set_seed(0)`, `set_seed(1)`, `set_seed(2)` before creating the model (and `seed=` in `train`). How much does the test accuracy vary? Does this match the "±1 point" rule of thumb?